# 3 · Basemaps & decoration

Web maps want context: a tile basemap underneath, coastlines/Natural-Earth features, a colorbar/legend
and text labels. These are the `tiles`, `coastlines`, `features`, `colorbar`/`legend`, `text` and `labels`
builders. Tile basemaps are **Web-Mercator only** (EPSG:3857), so these maps use `crs=3857`.

**Setup** — backends and the Rhine gauges.

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))

### `tiles` — an XYZ basemap
Drop a Carto basemap beneath the gauges. `list_tile_providers()` shows the catalog.

In [ ]:
m = InteractiveMap(crs=3857, title="gauges on a Carto basemap")
m.tiles("CartoLight").points(gauges, column="discharge", size=9, cmap="plasma")
show(m)

A few of the named providers available to `tiles(...)`:

In [ ]:
InteractiveMap().list_tile_providers()[:8]

### `coastlines` / `features` — Natural-Earth context
Overlay coastlines, or pick individual Natural-Earth features (land, ocean, borders, rivers, lakes). These load from the local Natural-Earth data package — no network needed.

In [ ]:
m = InteractiveMap(crs=3857, title="gauges with land + rivers")
m.features(land=True, rivers=True, lakes=True).points(gauges, size=7, color="crimson")
show(m)

### `labels` & `text` — annotation
`labels` places one text label per feature from a column; `text` stamps a single string at a lon/lat. Here we label the gauges by `name`.

In [ ]:
m = InteractiveMap(crs=3857, title="named gauges")
m.points(gauges, size=6, color="black")
m.labels(gauges, "name", text_font_size="7pt")
m.text(8.0, 49.5, "Rhine basin", text_font_size="12pt", text_color="navy")
show(m)

### `colorbar` / `legend` — a key that follows its layer
A colour key is a **guide on the layer's own colour encoding**, not a toggle on whichever layer was added
last. `colorbar(layer_id=None, *, label=None, visible=True)` and
`legend(layer_id=None, *, title=None, labels=None, visible=True)` record the guide and then draw it, so the
key moves with its layer, goes away with it, and is carried by the figure (`get_layer(id).symbology.guide()`
reads it back). `visible=False` drops the key; naming a `layer_id` describes that layer instead of the most
recent coloured one. A layer whose colour varies with nothing — a basemap, a coastline, a flat fill — is
refused a key rather than given an empty one.

In [ ]:
m = InteractiveMap(crs=3857, title="no colorbar")
m.points(gauges, column="discharge", size=9, cmap="plasma").colorbar(visible=False)
print(m.get_layer(m.layer_ids[0]).symbology.guide())
show(m)